In [1]:
import random
import numpy as np
import torch
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
# Set the random seed for reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
dataset = load_breast_cancer()
x, y = dataset.data, dataset.target
class_names = dataset.target_names

class_values, class_counts = np.unique(y, return_counts=True)
print(f"Number of examples: {x.shape[0]}")
print(f"Number of features: {x.shape[1]}")
print(f"Target classes: {class_names.tolist()}")
for value, count in zip(class_values, class_counts):
    print(f"Class {value} ({class_names[value]}): {count} examples")

Number of examples: 569
Number of features: 30
Target classes: ['malignant', 'benign']
Class 0 (malignant): 212 examples
Class 1 (benign): 357 examples


In [3]:
x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_valid = scaler.transform(x_valid)

x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
x_valid_tensor = torch.tensor(x_valid, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid, dtype=torch.long)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
valid_dataset = TensorDataset(x_valid_tensor, y_valid_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)

input_features = x_train.shape[1]
number_of_classes = len(class_names)

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(valid_dataset)}")
print(f"Training feature tensor shape: {x_train_tensor.shape}")
print(f"Validation feature tensor shape: {x_valid_tensor.shape}")
print(f"Number of classes: {number_of_classes}")

Training examples: 455
Validation examples: 114
Training feature tensor shape: torch.Size([455, 30])
Validation feature tensor shape: torch.Size([114, 30])
Number of classes: 2


In [4]:
class Model1(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Linear(16, output_size),
        )
        
    def forward(self, features):
        return self.network(features)
    
class Model2(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, output_size)
        )
        
    def forward(self, features):
        return self.network(features)
    
model_1 = Model1(input_features, number_of_classes).to(device)
model_2 = Model2(input_features, number_of_classes).to(device)
print(f"Model 1: \n{model_1}")
print(f"Model 2: \n{model_2}")

Model 1: 
Model1(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=2, bias=True)
  )
)
Model 2: 
Model2(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=2, bias=True)
  )
)


In [ ]:
def train_model(model, train_loader, epochs=50, learning_rate=0.001):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for features, targets in train_loader:
            features, targets = features.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = loss_function(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * features.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{epochs} - Training loss: {epoch_loss:.4f}")

print("Training Model 1")
train_model(model_1, train_loader)

print("\nTraining Model 2")
train_model(model_2, train_loader)